# TPC-H Collector Demo

# Missing
* connection data beyond loading
* join to connection data
* join to monitoring_single
* connection and configuration also in monitoring and connection df

## Naming
* There is an experiment having a code, say `1775855486`.
* The experiment inspects a SUT, say `PostgreSQL-A`. This is called a `configuration`.
* The experiment is run several times, say twice. The indicator of the run is called `experiment_run`.
* Each run can have several phases as a sequence. The number of the phase is called `client`. The state of the configuration in a phase is called a `connection`.
* Each client can have several `pods`, that are run in parallel. A pod represents a driver.
* Performance metrics are collected per driver pod.  
    The naming of an instance is `<sut>-<experiment_run>-<client>-<pod>`. It is unique per experiment.
* Monitoring metrics are collected per phase. They are automatically aggregated across parallel pods.  
    The naming of an instance is `<sut>-<experiment_run>-<client>`. It is unique per experiment.

## Aggregation
* Aggregation is complicated. Some metrics are aggregated via count, sum, max or average. Others cannot be aggregated sensibly, like experiment code or latency percentiles.
* There are helper functions to aggregated pods that are certainly run in parallel.  
  So `<sut>-<experiment_run>-<client>-<pod>` are aggregated to `<sut>-<experiment_run>-<client>`.
* An exception is multi-tenancy.

## Class `collector`
* constructor `collectors.benchbase(path, codes)`
* evaluator object `evaluate = collect.get_evaluator()`
* dataframe of connection infos `collect.get_connections()`
  * index is name of client
* **monitoring metrics**
  * dataframe of available metrics `collect.df_metrics`
    * index is key of metric
  * dataframe of monitored components `collect.get_monitored_phases()`
    * index is key of component
  * dataframe of aggregated metrics per connection `collect.get_monitoring_single_all()`
    * index is name of client
    * metrics aggregated per code, experiment_run and client
  * dataframe of aggregated metrics per connection and across tenants `collect.get_monitoring_all()`
    * index just enumerates
    * metrics aggregated per code, experiment_run and client and across tenants
  * dataframe of time series for a metric of a connection in an experiment `collect.get_monitoring_timeseries_single(code, metric)`
    * index is name of connection
    * unstacked (wide format)
  * dataframe of time series for a metric in all experiments `collect.get_monitoring_timeseries_all(metric)`
    * index just enumerates
    * stacked (long format)
* **performance metrics**
  * dataframe of loading metrics `collect.get_loading_time_max_all()`
    * index is name of client
  * dataframe of performance aggregated per parallel client `collect.get_performance_all()`
    * index just enumerates
    * performance aggregated per code, experiment_run and client
  * dataframe of performance for one experiment `collect.get_performance_single()`
    * performance of each single client (driver)
    * index is name of client pod
  * dataframe of performance for all experiments `collect.get_performance_all_single()`
    * performance of each single client (driver)
    * index is name of client pod


[1] [Benchmarking Multi-Tenant Architectures in PostgreSQL](https://doi.org/10.48786/edbt.2026.46)
> Erdelt, P.K., and Rabl T. (2026)
> In: Proceedings 29th International Conference on Extending Database Technology, EDBT 2026
> OpenProceedings.org
> https://doi.org/10.48786/edbt.2026.46


In [1]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option('display.max_colwidth', None)
pd.options.display.max_columns = None
pd.options.display.max_rows = None

from bexhoma import collectors

%matplotlib inline

# Functions for Nice Plots

In [2]:
from notebookutils import *

# Collect Results

In [3]:
path = r"D:\data\benchmarks"
#path = r"/data/benchmarks"
filename_prefix = "demo_"
b_plot_save = False
b_skip_plots = True

In [4]:
codes = [
    "1776508434",
    "1776550283"
]

codes

['1776508434', '1776550283']

In [5]:
collect = collectors.default(path, codes)

# Get all Metrics Metadata

## Metrics Names and Types

Metrics that are derived from monitoring

In [6]:
collect.df_metrics

,title,active,type,metric
total_cpu_memory,Memory Usage [MiB],True,cluster,gauge
total_cpu_memory_cached,Memory Usage Cached [MiB],True,cluster,gauge
total_cpu_util,CPU Utilization,True,cluster,gauge
total_cpu_throttled,CPU Throttle,True,cluster,gauge
total_cpu_throttled_s,CPU Throttled Time [s],True,cluster,counter
total_cpu_util_others,CPU Utilization Others,False,cluster,gauge
total_cpu_util_s,CPU Utilization Time [s],True,cluster,counter
total_cpu_util_user_s,CPU User Time [s],True,cluster,counter
total_cpu_util_sys_s,CPU System Time [s],True,cluster,counter
total_cpu_util_others_s,CPU Utilization Time Others [s],False,cluster,counter


## Names of Monitored Phases

Names of components and their phases

In [7]:
collect.get_monitored_components()

,description
loading,Loading phase: SUT deployment
datagenerator,Loading phase: component data generator
loader,Loading phase: component loader
stream,Execution phase: SUT deployment
benchmarker,Execution phase: component benchmarker


# Get Connection Infos

List of states of SUTs, containing loading infos.

In [8]:
df = collect.get_connections()
df.T

,1776508434-PostgreSQL-BHT-8-1-1,1776508434-PostgreSQL-BHT-8-1-2,1776508434-PostgreSQL-BHT-8-2-1,1776508434-PostgreSQL-BHT-8-2-2,1776550283-PostgreSQL-BHT-8-1-1,1776550283-PostgreSQL-BHT-8-1-2,1776550283-PostgreSQL-BHT-8-2-1,1776550283-PostgreSQL-BHT-8-2-2
code,1776508434,1776508434,1776508434,1776508434,1776550283,1776550283,1776550283,1776550283
connection,PostgreSQL-BHT-8-1-1-1,PostgreSQL-BHT-8-1-2-2,PostgreSQL-BHT-8-2-1-1,PostgreSQL-BHT-8-2-2-2,PostgreSQL-BHT-8-1-1-1,PostgreSQL-BHT-8-1-2-2,PostgreSQL-BHT-8-2-1-1,PostgreSQL-BHT-8-2-2-2
configuration,,,,,PostgreSQL-BHT-8,PostgreSQL-BHT-8,PostgreSQL-BHT-8,PostgreSQL-BHT-8
phase,1776508434-PostgreSQL-BHT-8-1-1,1776508434-PostgreSQL-BHT-8-1-2,1776508434-PostgreSQL-BHT-8-2-1,1776508434-PostgreSQL-BHT-8-2-2,1776550283-PostgreSQL-BHT-8-1-1,1776550283-PostgreSQL-BHT-8-1-2,1776550283-PostgreSQL-BHT-8-2-1,1776550283-PostgreSQL-BHT-8-2-2
experiment_run,1,1,2,2,1,1,2,2
client,1,2,1,2,1,2,1,2
dockerimage,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3
time_load,49824.0,49824.0,49824.0,49824.0,114757.0,114757.0,114757.0,114757.0
time_ingest,22553.0,22553.0,22553.0,22553.0,52684.0,52684.0,52684.0,52684.0
time_check,27246.0,27246.0,27246.0,27246.0,62049.0,62049.0,62049.0,62049.0


# Get Aggregated Metrics per SUT and per Experiment

In [9]:
df_performance = collect.get_monitoring_aggregated_per_phase("stream")
df_performance.T

,1776508434-PostgreSQL-BHT-8-1-1,1776508434-PostgreSQL-BHT-8-1-2,1776508434-PostgreSQL-BHT-8-2-1,1776508434-PostgreSQL-BHT-8-2-2,1776550283-PostgreSQL-BHT-8-1-1,1776550283-PostgreSQL-BHT-8-1-2,1776550283-PostgreSQL-BHT-8-2-1,1776550283-PostgreSQL-BHT-8-2-2
Memory Usage [MiB],10126.29,11046.67,10436.84,10392.82,16778.00,20248.53,13184.61,17880.38
Memory Usage Cached [MiB],15043.22,15963.60,17300.41,16850.84,26572.73,30043.26,25182.63,29738.30
CPU Utilization,1.60,3.12,2.26,3.05,2.34,4.57,2.68,6.16
CPU Throttle,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
CPU Throttled Time [s],0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
CPU Utilization Time [s],134.77,284.83,329.93,204.73,402.12,749.50,692.74,874.70
CPU User Time [s],118.61,251.22,304.68,182.47,354.00,665.73,624.44,774.79
CPU System Time [s],16.16,33.61,25.25,22.26,48.12,83.77,68.30,99.92
Network Rx Total [MiB],1.86,0.51,1.74,1.88,2.21,4.07,1.92,3.74
Network Tx Total [MiB],2.85,4.49,1.72,3.35,7.95,9.88,1.91,3.73


In [10]:
df_performance = collect.add_metadata(df_performance)
df_performance.T

combine on index and column 'phase'


phase,1776508434-PostgreSQL-BHT-8-1-1,1776508434-PostgreSQL-BHT-8-1-2,1776508434-PostgreSQL-BHT-8-2-1,1776508434-PostgreSQL-BHT-8-2-2,1776550283-PostgreSQL-BHT-8-1-1,1776550283-PostgreSQL-BHT-8-1-2,1776550283-PostgreSQL-BHT-8-2-1,1776550283-PostgreSQL-BHT-8-2-2
Memory Usage [MiB],10126.29,11046.67,10436.84,10392.82,16778.0,20248.53,13184.61,17880.38
Memory Usage Cached [MiB],15043.22,15963.6,17300.41,16850.84,26572.73,30043.26,25182.63,29738.3
CPU Utilization,1.6,3.12,2.26,3.05,2.34,4.57,2.68,6.16
CPU Throttle,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CPU Throttled Time [s],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CPU Utilization Time [s],134.77,284.83,329.93,204.73,402.12,749.5,692.74,874.7
CPU User Time [s],118.61,251.22,304.68,182.47,354.0,665.73,624.44,774.79
CPU System Time [s],16.16,33.61,25.25,22.26,48.12,83.77,68.3,99.92
Network Rx Total [MiB],1.86,0.51,1.74,1.88,2.21,4.07,1.92,3.74
Network Tx Total [MiB],2.85,4.49,1.72,3.35,7.95,9.88,1.91,3.73


# Bar Plots of Aggregated Values per Metric

In [11]:
metric = 'total_cpu_memory'
code = codes[0]
df = collect.get_monitoring_timeseries_per_phase(code, metric=metric)
df.T

,1776508434-PostgreSQL-BHT-8-1-1,1776508434-PostgreSQL-BHT-8-1-2,1776508434-PostgreSQL-BHT-8-2-1,1776508434-PostgreSQL-BHT-8-2-2
0,9757.070312,10318.062500,9749.792969,10312.445312
1,9757.070312,10318.062500,9749.792969,10312.445312
2,9757.070312,10318.062500,9749.792969,10312.445312
3,9757.070312,10318.062500,9749.792969,10312.445312
4,9757.070312,10318.062500,9749.792969,10312.445312
5,9757.070312,10318.062500,9749.792969,10312.445312
6,9757.070312,10318.062500,9749.792969,10312.445312
7,9757.070312,10318.062500,9838.878906,10312.445312
8,9757.070312,10318.062500,9838.878906,10312.445312
9,9757.070312,10318.062500,9838.878906,10312.445312


In [12]:
df_performance = collect.add_metadata(df)
df_performance.T

combine on index and column 'phase'


phase,1776508434-PostgreSQL-BHT-8-1-1,1776508434-PostgreSQL-BHT-8-1-2,1776508434-PostgreSQL-BHT-8-2-1,1776508434-PostgreSQL-BHT-8-2-2
0,9757.070312,10318.0625,9749.792969,10312.445312
1,9757.070312,10318.0625,9749.792969,10312.445312
2,9757.070312,10318.0625,9749.792969,10312.445312
3,9757.070312,10318.0625,9749.792969,10312.445312
4,9757.070312,10318.0625,9749.792969,10312.445312
5,9757.070312,10318.0625,9749.792969,10312.445312
6,9757.070312,10318.0625,9749.792969,10312.445312
7,9757.070312,10318.0625,9838.878906,10312.445312
8,9757.070312,10318.0625,9838.878906,10312.445312
9,9757.070312,10318.0625,9838.878906,10312.445312


In [13]:
metric = 'total_cpu_memory'
code = codes[0]
collect.get_monitoring_timeseries_all(metric=metric)

,timestamp,code,phase,experiment_run,client,type_tenants,vol_tenants,num_tenants,metric,component,value
0,0,1776508434,1776508434-PostgreSQL-BHT-8-1-1,1,1,,False,0,total_cpu_memory,stream,9757.070312
1,0,1776508434,1776508434-PostgreSQL-BHT-8-1-2,1,2,,False,0,total_cpu_memory,stream,10318.062500
2,0,1776508434,1776508434-PostgreSQL-BHT-8-2-1,2,1,,False,0,total_cpu_memory,stream,9749.792969
3,0,1776508434,1776508434-PostgreSQL-BHT-8-2-2,2,2,,False,0,total_cpu_memory,stream,10312.445312
4,0,1776550283,1776550283-PostgreSQL-BHT-8-1-1,1,1,,False,0,total_cpu_memory,stream,13747.460938
5,0,1776550283,1776550283-PostgreSQL-BHT-8-1-2,1,2,,False,0,total_cpu_memory,stream,14869.750000
6,0,1776550283,1776550283-PostgreSQL-BHT-8-2-1,2,1,,False,0,total_cpu_memory,stream,5888.609375
7,0,1776550283,1776550283-PostgreSQL-BHT-8-2-2,2,2,,False,0,total_cpu_memory,stream,14751.621094
8,1,1776508434,1776508434-PostgreSQL-BHT-8-1-1,1,1,,False,0,total_cpu_memory,stream,9757.070312
9,1,1776508434,1776508434-PostgreSQL-BHT-8-1-2,1,2,,False,0,total_cpu_memory,stream,10318.062500


In [18]:
df_performance = collect.get_performance_per_connection()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.dropna(inplace=True)
df_performance.T

,1776508434-PostgreSQL-BHT-8-1-1-1,1776508434-PostgreSQL-BHT-8-1-2-1,1776508434-PostgreSQL-BHT-8-1-2-2,1776508434-PostgreSQL-BHT-8-2-1-1,1776508434-PostgreSQL-BHT-8-2-2-1,1776508434-PostgreSQL-BHT-8-2-2-2,1776550283-PostgreSQL-BHT-8-1-1-1,1776550283-PostgreSQL-BHT-8-1-2-1,1776550283-PostgreSQL-BHT-8-1-2-2,1776550283-PostgreSQL-BHT-8-2-1-1,1776550283-PostgreSQL-BHT-8-2-2-1,1776550283-PostgreSQL-BHT-8-2-2-2
total_timer_execution,1.569886,1.580692,1.547605,2.203745,1.558278,1.561981,3.026814,3.050032,3.053193,4.250588,3.034907,3.074278
Power@Size [~Q/h],6879.478867,6832.452394,6978.523856,4900.749464,6930.729424,6914.296866,7136.215611,7081.892596,7074.561948,5081.650437,7117.18614,7026.038898
Geo Times [s],1.569886,1.580692,1.547605,2.203745,1.558278,1.561981,3.026814,3.050032,3.053193,4.250588,3.034907,3.074278
configuration,1776508434--,1776508434--,1776508434--,1776508434--,1776508434--,1776508434--,1776550283-PostgreSQL-BHT-8,1776550283-PostgreSQL-BHT-8,1776550283-PostgreSQL-BHT-8,1776550283-PostgreSQL-BHT-8,1776550283-PostgreSQL-BHT-8,1776550283-PostgreSQL-BHT-8
connection,1776508434-PostgreSQL-BHT-8-1-1-1,1776508434-PostgreSQL-BHT-8-1-2-1,1776508434-PostgreSQL-BHT-8-1-2-2,1776508434-PostgreSQL-BHT-8-2-1-1,1776508434-PostgreSQL-BHT-8-2-2-1,1776508434-PostgreSQL-BHT-8-2-2-2,1776550283-PostgreSQL-BHT-8-1-1-1,1776550283-PostgreSQL-BHT-8-1-2-1,1776550283-PostgreSQL-BHT-8-1-2-2,1776550283-PostgreSQL-BHT-8-2-1-1,1776550283-PostgreSQL-BHT-8-2-2-1,1776550283-PostgreSQL-BHT-8-2-2-2
phase,1776508434-PostgreSQL-BHT-8-1-1,1776508434-PostgreSQL-BHT-8-1-2,1776508434-PostgreSQL-BHT-8-1-2,1776508434-PostgreSQL-BHT-8-2-1,1776508434-PostgreSQL-BHT-8-2-2,1776508434-PostgreSQL-BHT-8-2-2,1776550283-PostgreSQL-BHT-8-1-1,1776550283-PostgreSQL-BHT-8-1-2,1776550283-PostgreSQL-BHT-8-1-2,1776550283-PostgreSQL-BHT-8-2-1,1776550283-PostgreSQL-BHT-8-2-2,1776550283-PostgreSQL-BHT-8-2-2
SF,3.0,3.0,3.0,3.0,3.0,3.0,6.0,6.0,6.0,6.0,6.0,6.0
experiment_run,1,1,1,2,2,2,1,1,1,2,2,2
client,1,2,2,1,2,2,1,2,2,1,2,2
time [s],63,61,60,108,60,62,116,122,120,209,118,118


In [15]:
df_performance = collect.get_performance_aggregated_per_phase()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

,1776508434-PostgreSQL-BHT-8-1-1-1,1776508434-PostgreSQL-BHT-8-1-2-1,1776508434-PostgreSQL-BHT-8-1-2-2,1776508434-PostgreSQL-BHT-8-2-1-1,1776508434-PostgreSQL-BHT-8-2-2-1,1776508434-PostgreSQL-BHT-8-2-2-2,1776550283-PostgreSQL-BHT-8-1-1-1,1776550283-PostgreSQL-BHT-8-1-2-1,1776550283-PostgreSQL-BHT-8-1-2-2,1776550283-PostgreSQL-BHT-8-2-1-1,1776550283-PostgreSQL-BHT-8-2-2-1,1776550283-PostgreSQL-BHT-8-2-2-2
configuration,1776508434--,1776508434--,1776508434--,1776508434--,1776508434--,1776508434--,1776550283-PostgreSQL-BHT-8,1776550283-PostgreSQL-BHT-8,1776550283-PostgreSQL-BHT-8,1776550283-PostgreSQL-BHT-8,1776550283-PostgreSQL-BHT-8,1776550283-PostgreSQL-BHT-8
experiment_run,1,1,1,2,2,2,1,1,1,2,2,2
phase,1776508434-PostgreSQL-BHT-8-1-1,1776508434-PostgreSQL-BHT-8-1-2,1776508434-PostgreSQL-BHT-8-1-2,1776508434-PostgreSQL-BHT-8-2-1,1776508434-PostgreSQL-BHT-8-2-2,1776508434-PostgreSQL-BHT-8-2-2,1776550283-PostgreSQL-BHT-8-1-1,1776550283-PostgreSQL-BHT-8-1-2,1776550283-PostgreSQL-BHT-8-1-2,1776550283-PostgreSQL-BHT-8-2-1,1776550283-PostgreSQL-BHT-8-2-2,1776550283-PostgreSQL-BHT-8-2-2
total_timer_execution,1.569886,1.580692,1.547605,2.203745,1.558278,1.561981,3.026814,3.050032,3.053193,4.250588,3.034907,3.074278
Power@Size [~Q/h],6879.478867,6832.452394,6978.523856,4900.749464,6930.729424,6914.296866,7136.215611,7081.892596,7074.561948,5081.650437,7117.18614,7026.038898
code,1776508434,1776508434,1776508434,1776508434,1776508434,1776508434,1776550283,1776550283,1776550283,1776550283,1776550283,1776550283
pod_count,1,1,1,1,1,1,1,1,1,1,1,1
SF,3.0,3.0,3.0,3.0,3.0,3.0,6.0,6.0,6.0,6.0,6.0,6.0
time [s],63,61,60,108,60,62,116,122,120,209,118,118
client,1,2,2,1,2,2,1,2,2,1,2,2


In [16]:
df = collect.add_metadata(df_performance)
df.T


combine on columns phase


,1776508434-PostgreSQL-BHT-8-1-1,1776508434-PostgreSQL-BHT-8-1-2,1776508434-PostgreSQL-BHT-8-1-2,1776508434-PostgreSQL-BHT-8-2-1,1776508434-PostgreSQL-BHT-8-2-2,1776508434-PostgreSQL-BHT-8-2-2,1776550283-PostgreSQL-BHT-8-1-1,1776550283-PostgreSQL-BHT-8-1-2,1776550283-PostgreSQL-BHT-8-1-2,1776550283-PostgreSQL-BHT-8-2-1,1776550283-PostgreSQL-BHT-8-2-2,1776550283-PostgreSQL-BHT-8-2-2
Power@Size [~Q/h],6879.478867,6832.452394,6978.523856,4900.749464,6930.729424,6914.296866,7136.215611,7081.892596,7074.561948,5081.650437,7117.18614,7026.038898
SF,3.0,3.0,3.0,3.0,3.0,3.0,6.0,6.0,6.0,6.0,6.0,6.0
Throughput@Size,3771.43,3895.08,3960.0,2200.0,3960.0,3832.26,4096.55,3895.08,3960.0,2273.68,4027.12,4027.12
arg_autovacuum,off,off,off,off,off,off,off,off,off,off,off,off
arg_checkpoint_completion_target,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
arg_checkpoint_timeout,12h,12h,12h,12h,12h,12h,12h,12h,12h,12h,12h,12h
arg_effective_cache_size,256GB,256GB,256GB,256GB,256GB,256GB,256GB,256GB,256GB,256GB,256GB,256GB
arg_effective_io_concurrency,64,64,64,64,64,64,64,64,64,64,64,64
arg_fsync,on,on,on,on,on,on,on,on,on,on,on,on
arg_io_method,worker,worker,worker,worker,worker,worker,worker,worker,worker,worker,worker,worker


# Boxplot of All Metrics for a Single Experiment

In [ ]:
if not b_skip_plots:
    results = []
    code = codes[0]
    for idx, row in collect.df_metrics.iterrows():
        if row["active"] == False:
            continue
        metric_name = idx
        method = 'diff' if row["metric"] == 'counter' else 'mean'
        col_name = row["title"]
        df_monitoring = collect.get_monitoring_timeseries_single(code, metric=metric_name)
        #print(df_monitoring)
        ax = df_monitoring.boxplot()
        ax.set_title(col_name)
        plt.show()

# Boxplot of A Single Metric for a Single Experiment

In [ ]:
metric = 'total_cpu_memory'
code = codes[0]
collect.get_monitoring_timeseries_single(code, metric=metric)

In [ ]:
metric = 'total_cpu_memory'
code = codes[0]
collect.get_monitoring_timeseries_all( metric=metric)

In [ ]:
if not b_skip_plots:
    metric = 'pg_stat_activity_max_tx_duration_active'
    #metric = 'pg_stat_database_blks_reads'
    #metric = 'pg_stat_activity_count_idle_transaction'
    metric = 'total_cpu_memory'
    code = codes[0]
    df_monitoring = collect.get_monitoring_timeseries_single(code, metric=metric)
    ax = df_monitoring.boxplot()
    ax.set_title(collect.df_metrics.loc[metric]['title'])#metric)
    plt.show()
    df_monitoring

# Lineplot of a Single Metric for a Single Experiment

In [ ]:
if not b_skip_plots:
    code = codes[0]
    metric = 'pg_stat_database_blks_hit'
    metric = 'pg_stat_activity_count_idle_transaction'
    metric = 'total_cpu_memory'
    metric = 'total_cpu_util'
    metric = 'total_cpu_util_s'
    metric = 'core_variance'
    df_monitoring = collect.get_monitoring_timeseries_single(code, metric=metric)
    ax = df_monitoring.plot()
    ax.set_title(metric)
    plt.show()
    #df_monitoring

# Boxplot of a Single Metric for all Experiments

In [ ]:
metric = 'pg_stat_database_blks_hit'
metric = 'pg_stat_activity_count_idle_transaction'
metric = 'total_cpu_memory'
metric = 'total_cpu_util'
metric = 'total_cpu_util_s'
metric = 'core_variance'
metric = 'pg_locks_count_exclusivelock'
metric = 'pg_stat_activity_count_active'
df_performance = collect.get_monitoring_timeseries_all(metric)
#df_performance
df_performance_first = df_performance[df_performance['client']=="1"]
#df_performance_first
df_performance_second = df_performance[df_performance['client']=="2"]
#df_performance_second
#df_performance_first['value'].describe()
#df_performance

## First Execution Run

In [ ]:
plot_boxplots(df_performance_first, y='value', title=collect.df_metrics.loc[metric]['title'], b_plot_save=b_plot_save, filename_prefix=filename_prefix)

## Second Execution Run

In [ ]:
plot_boxplots(df_performance_second, y='value', title=collect.df_metrics.loc[metric]['title'], b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Lineplot of a Single Metric for All Experiments

## First Execution Run

In [ ]:
#df = df_performance_first[df_performance_first['type'] == 'None']
#df = df[df['experiment_run'] == '1']
plot_lines(df_performance_first, y='value', title=collect.df_metrics.loc[metric]['title'], b_plot_save=b_plot_save, filename_prefix=filename_prefix)
#df

# Aggregated Metrics for all Connections

In [ ]:
df_performance = collect.get_monitoring_single_all("stream")
df_performance.T

# Monitoring Aggregated Values

In [ ]:
df_performance = collect.get_monitoring_all("stream")

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T#[['Max CPU', 'client', 'type', 'num_tenants']]

## First Execution Run

In [ ]:
plot_bars(df_performance_first, y='Buffer Cache Hit Ratio', title='Cache Hit Ratio [%]', estimator='min', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

## Second Execution Run

In [ ]:
plot_bars(df_performance_second, y='Buffer Cache Hit Ratio', title='Cache Hit Ratio [%]', estimator='min', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Performance Results per Tenant

In [ ]:
df_performance = collect.get_performance_all_single()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

## First Execution Run

In [ ]:
plot_boxplots(df_performance_first, y='Goodput (requests/second)', title='Distribution of Goodput First Run [req/s]', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_boxplots(df_performance_second, y='Goodput (requests/second)', title='Distribution of Goodput Second Run [req/s]', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_boxplots(df_performance_first, y='Latency Distribution.99th Percentile Latency (microseconds)', title=r'Distribution of 99th Latency 1st Run [$\mu s$]', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_boxplots(df_performance_second, y='Latency Distribution.99th Percentile Latency (microseconds)', title=r'Distribution of 99th Latency 2nd Run [$\mu s$]', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Performance Results per Total

In [ ]:
collect.get_performance_single().T

In [ ]:
collect.get_performance_all_single().T

In [ ]:
df_performance = collect.get_performance_all()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.dropna(inplace=True)

In [ ]:
df_performance.T

In [ ]:
import seaborn as sns

sns.barplot(data=df_performance, x='experiment_run', y='Goodput (requests/second)', hue='client')
plt.show()


In [ ]:
plot_bars(df_performance, y='Goodput (requests/second)', title='Goodput [req/s]', estimator='min', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_first, y='Goodput (requests/second)', title='Goodput First Run [req/s]', estimator='min', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_second, y='Goodput (requests/second)', title='Goodput Second Run [req/s]', estimator='min', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance, y='Latency Distribution.Average Latency (microseconds)', title=r'Average Latency [$\mu s$]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_first, y='Latency Distribution.Average Latency (microseconds)', title=r'Average Latency First Run [$\mu s$]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_second, y='Latency Distribution.Average Latency (microseconds)', title=r'Average Latency Second Run [$\mu s$]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance, y='num_errors', title='Deadlocks', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
df_performance = collect.get_loading_time_max_all()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance

In [ ]:
df_performance_first = df_performance[df_performance['client'] == 1]
# Divide datadisk by the count of rows with the same type and num_tenants
df = df_performance_first.copy()
# Create a mask for rows where type is not "container"
mask = df['type'] != 'container'

# Only apply the group count to the relevant rows
group_counts = df[mask].groupby(['type', 'num_tenants'])['datadisk'].transform('count')

# Initialize the column with NaN (or 0, if preferred)
df['datadisk_normalized'] = df['datadisk'] / 1024

# Apply the normalized value only where the mask is True
df.loc[mask, 'datadisk_normalized'] = df.loc[mask, 'datadisk'] / group_counts / 1024

plot_bars(df, y='datadisk_normalized', title='Database Size [GB]', estimator='sum', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_first, y='datadisk', title='Database Size [MB]', estimator='sum', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_first, y='time_ingest', title='Time for Ingestion [s]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_first, y='time_check', title='Time for Vacuum [s]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Hardware Monitoring for Benchmarking Phase

In [ ]:
df_performance = collect.get_monitoring_all(type="stream")

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

In [ ]:
plot_bars(df_performance, y='CPU Utilization Time [s]', title='CPU [CPUs]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_first, y='CPU Utilization Time [s]', title='CPU 1st Run [CPUs]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_second, y='CPU Utilization Time [s]', title='CPU 2nd Run [CPUs]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance, y='CPU Utilization', title='CPU Utilization [%]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

## Custom Aggregation

In [ ]:
metric = 'total_cpu_util'

df_performance_series = collect.get_monitoring_timeseries_all(metric)

df_agg = (
    df_performance_series.groupby(["client", "type", "num_tenants"])["value"]
      .max()
      .reset_index()
)
plot_bars(df_agg, y='value', title='Max CPU Utilization [%]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)
#df_agg

In [ ]:
plot_bars(df_performance, y='CPU Throttle', title='CPU Throttle [%]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

## Custom Aggregation and Scale

In [ ]:
metric = 'total_cpu_memory'

df_performance_series = collect.get_monitoring_timeseries_all(metric)

df_agg = (
    df_performance_series.groupby(["client", "type", "num_tenants"])["value"]
      .max()
      .reset_index()
)
df_agg['value'] = df_agg['value'] / 1024.
plot_bars(df_agg, y='value', title='Max Memory Usage [GiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

plot_bars(df_performance, y='Memory Usage [MiB]', title='Average Memory Usage [MiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
metric = 'total_cpu_memory_cached'

df_performance_series = collect.get_monitoring_timeseries_all(metric)

df_agg = (
    df_performance_series.groupby(["client", "type", "num_tenants"])["value"]
      .max()
      .reset_index()
)
df_agg['value'] = df_agg['value'] / 1024.
plot_bars(df_agg, y='value', title='Max Memory Usage Cached [GiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

plot_bars(df_performance, y='Memory Usage Cached [MiB]', title='Average Memory Usage Cached [MiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Hardware Monitoring for Loading Phase

In [ ]:
df_performance = collect.get_monitoring_all("loading")

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance_first

In [ ]:
plot_bars(df_performance, y='CPU Utilization Time [s]', title='CPU [CPUs]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance, y='Memory Usage [MiB]', title='Memory Usage [MiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance, y='Memory Usage Cached [MiB]', title='Memory Usage Cached [MiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Efficiency

In [ ]:
client = 1

df_performance_monitoring = collect.get_monitoring_all(type="stream")
df_performance_monitoring = df_performance_monitoring[df_performance_monitoring['client'] == client]
df_performance = collect.get_performance_all()
df_performance = df_performance[df_performance['client'] == client]
merged_df = pd.merge(df_performance, df_performance_monitoring, on=['type', 'num_tenants', 'code', 'client'], how='inner')
#merged_df['I_Lat'] = 1./merged_df['E_Lat']
merged_df['E_Tpx'] = merged_df['Goodput (requests/second)'] / merged_df['CPU Utilization Time [s]'] * 600.
merged_df['E_Lat'] = 1./np.sqrt(merged_df['Latency Distribution.Average Latency (microseconds)']*merged_df['CPU Utilization Time [s]']/1E6)
merged_df['E_RAM'] = (merged_df['Goodput (requests/second)']) / merged_df['Memory Usage [MiB]']
merged_df

In [ ]:
plot_bars(merged_df, y='E_Lat', title='1st run - $E_{Lat}$', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
#plot_bars(merged_df, y='I_Lat', title='1st run - $I_{Lat}$', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(merged_df, y='E_Tpx', title='1st run - $E_{Tpx}$', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(merged_df, y='E_RAM', title='1st run - $E_{RAM}$', estimator='min', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
client = 2

df_performance_monitoring = collect.get_monitoring_all(type="stream")
df_performance_monitoring = df_performance_monitoring[df_performance_monitoring['client'] == client]
df_performance = collect.get_performance_all()
df_performance = df_performance[df_performance['client'] == client]
merged_df = pd.merge(df_performance, df_performance_monitoring, on=['type', 'num_tenants', 'code', 'client'], how='inner')
#merged_df['CPUs/Request'] = merged_df['CPU [CPUs]'] / merged_df['Goodput (requests/second)'] / 600.
merged_df['E_Tpx'] = merged_df['Goodput (requests/second)'] / merged_df['CPU Utilization Time [s]'] * 600.
merged_df['E_Lat'] = 1./np.sqrt(merged_df['Latency Distribution.Average Latency (microseconds)']*merged_df['CPU Utilization Time [s]']/1E6)
merged_df['E_RAM'] = (merged_df['Goodput (requests/second)']) / merged_df['Memory Usage [MiB]']

merged_df

In [ ]:
plot_bars(merged_df, y='E_Lat', title='2nd run - $E_{Lat}$', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(merged_df, y='E_Tpx', title='2nd run - $E_{Tpx}$', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(merged_df, y='E_RAM', title='2nd run - $E_{RAM}$', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
#zip_all_results()